# 0.10 · 数值优化 / Numerical Optimization

> **课程定位 / Where this fits**
> 第 10 课，**Part 0 · 基础准备**。
> Lesson 10, **Part 0 · Foundations**.
>
> 0.8 节我们用了基础梯度下降。这一节把它**升级为现代优化体系**：凸性 → KKT → Newton → Adam 家族。理解这套是 **PyTorch 选 optimizer**、**调 lr scheduler**、**写自定义训练循环**的基础。
> Lesson 0.8 introduced GD. This lesson upgrades it to the full modern stack: convexity → KKT → Newton → Adam family. The basis for everything that picks an optimizer or writes a custom training loop.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $f, J$ —— 目标函数 / objective
> - $\mathbf{x}^*$ —— 最优解 / optimum
> - $\mathbf{g} = \nabla f$ —— 梯度
> - $\mathbf{H} = \nabla^2 f$ —— Hessian
> - $\eta$ —— 学习率
> - $\mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\mu})$ —— 拉格朗日函数 / Lagrangian

> 💡 **面试相关 / Interview-relevant**
> - "为什么 SGD 比 Newton 在 DL 里好用" ★★★★★
> - "Adam vs SGD with momentum" ★★★★★
> - 凸函数定义 / KKT 三句话答 ★★★
> - "什么时候用 L-BFGS" ★★★
>
> Top hits: SGD-vs-Newton tradeoff, Adam-vs-SGD-momentum, convex definitions, KKT, L-BFGS use cases.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 准确说出**凸集 / 凸函数**的定义，并知道凸性带来什么 ML 优势。
   State the definitions of convex sets / functions and what convexity buys you in ML.
2. 写出无约束 / 等式约束 / 不等式约束的**最优条件**（含 **KKT**）。
   Write down optimality conditions for unconstrained / equality / inequality cases (incl. **KKT**).
3. 解释 **Newton 法**为什么"步长方向更聪明"，并知道它的瓶颈在哪。
   Explain why Newton's method takes "smarter steps" and where it breaks down.
4. 区分 **SGD / Momentum / Nesterov / Adagrad / RMSProp / Adam / AdamW** 的更新规则差异。
   Tell apart the update rules of the modern first-order optimizers.
5. 选**对的优化器和学习率调度**应对一个新问题。
   Pick the right optimizer + LR scheduler for a new problem.
6. 在二分类问题上**用 Newton 法手写 Logistic 回归**，并和 GD 对比收敛速度。
   Implement logistic regression with Newton's method and compare to GD.

---

## 目录 / Table of Contents

1. [优化问题的标准形式 / Standard Form](#1)
2. [凸集 & 凸函数 ⭐ / Convex Sets & Functions](#2)
3. [无约束最优条件 / Unconstrained Optimality](#3)
4. [拉格朗日 & KKT ⭐ / Lagrangian & KKT](#4)
5. [Newton 法 / Newton's Method](#5)
6. [拟 Newton：BFGS & L-BFGS](#6)
7. [一阶方法谱系 / First-Order Optimizer Zoo](#7)
8. [学习率调度 / LR Schedulers](#8)
9. [实战 1：Newton 法 vs GD 解 Logistic 回归 / Hands-on](#9)
10. [实战 2：六种优化器在 2D 函数上对比 / Optimizer Race](#10)
11. [小结 / Summary](#11)


<a id="1"></a>
## 1. 优化问题的标准形式 / Standard Form

$$
\begin{aligned}
\min_{\mathbf{x} \in \mathbb{R}^n} \;\; & f(\mathbf{x}) \\
\text{s.t.} \;\; & g_i(\mathbf{x}) \le 0, \quad i = 1, \dots, m \\
& h_j(\mathbf{x}) = 0, \quad j = 1, \dots, p
\end{aligned}
$$

- $f$ —— 目标函数 / objective
- $g_i \le 0$ —— 不等式约束 / inequality constraints
- $h_j = 0$ —— 等式约束 / equality constraints

**机器学习里 90% 的训练问题是无约束** ($m = p = 0$)，但**正则化 + Lasso + SVM** 都涉及约束。
Most ML training is unconstrained, but regularization / Lasso / SVM bring constraints back.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import scipy.optimize as opt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)


<a id="2"></a>
## 2. 凸集 & 凸函数 ⭐ / Convex Sets & Functions

### 2.1 凸集 / Convex set

集合 $C \subseteq \mathbb{R}^n$ **凸** ⇔ 任意两点连线都在 $C$ 内：
$$\forall \mathbf{x}, \mathbf{y} \in C, \;\; \forall t \in [0, 1]: \;\; t\mathbf{x} + (1-t)\mathbf{y} \in C$$

例：球、半空间、超平面、$\ell_p$ 球（$p \ge 1$）。**非凸**：环、星形、$\ell_p$ 球（$0 < p < 1$）。

### 2.2 凸函数 / Convex function

$f: \mathbb{R}^n \to \mathbb{R}$ **凸** ⇔
$$\forall \mathbf{x}, \mathbf{y}, \; \forall t \in [0,1]:\;\; f(t\mathbf{x} + (1-t)\mathbf{y}) \le t f(\mathbf{x}) + (1-t)f(\mathbf{y})$$

**几何**：图像上任意两点的弦总在图像之上。
**Geometric**: any chord lies above the graph.

### 2.3 凸性带来的 ML 神兵 / Why ML loves convexity

- 任何**局部最小 = 全局最小** / Every local minimum is global ⭐
- 一阶必要条件 $\nabla f(\mathbf{x}^*) = \mathbf{0}$ 立即变成**充分**条件
- 算法收敛保证（GD、Newton、内点法都能给出收敛速度）

### 2.4 二阶判定 / Second-order check

$f$ 二阶连续可微：
$$f \text{ 凸} \;\iff\; \mathbf{H}(\mathbf{x}) \succeq 0 \;\forall \mathbf{x}$$

即 Hessian **半正定**（所有特征值 $\ge 0$）。
i.e. Hessian is positive semi-definite at every $\mathbf{x}$.

### 2.5 ML 里**凸**的损失 / Convex losses in ML

| 损失 / Loss | 凸吗 / Convex? |
|---|---|
| 线性回归 MSE $\|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2$ | ✅ |
| 逻辑回归 cross-entropy | ✅ |
| SVM hinge loss | ✅ |
| Lasso $\|\mathbf{w}\|_1$ | ✅ (convex but non-smooth) |
| L2 正则 $\|\mathbf{w}\|^2$ | ✅ 且强凸 / strongly convex |
| **神经网络** 总损失 | ❌ **非凸**！但 SGD 仍然work |

> 💡 **DL 的反直觉 / DL paradox**：NN 损失非凸但实际可解，因为高维下"差不多好"的最小值无处不在，SGD 找到一个就够了。
> NN losses are non-convex but trainable — high dimensions create plenty of "good enough" minima.


In [ ]:
# 可视化凸 vs 非凸函数 / Visualize convex vs non-convex
xs = np.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# 1) 凸
axes[0].plot(xs, xs**2, "b-", linewidth=2)
axes[0].plot([-2, 2], [4, 4], "r--", label="chord")
axes[0].set_title("f(x) = x²    convex ✅")
axes[0].legend()

# 2) 凸但非光滑 / convex non-smooth
axes[1].plot(xs, np.abs(xs), "b-", linewidth=2)
axes[1].plot([-2, 1.5], [2, 1.5], "r--", label="chord")
axes[1].set_title("f(x) = |x|   convex (non-smooth) ✅")
axes[1].legend()

# 3) 非凸 / non-convex
axes[2].plot(xs, np.sin(2*xs) + 0.1*xs**2, "b-", linewidth=2)
axes[2].axhline(0, color="gray", lw=0.5)
axes[2].set_title("f(x) = sin(2x)+0.1x²   non-convex ❌\nmultiple local min")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


<a id="3"></a>
## 3. 无约束最优条件 / Unconstrained Optimality

### 一阶必要条件 / First-order necessary

若 $\mathbf{x}^*$ 是 $f$ 的局部极小，且 $f$ 在 $\mathbf{x}^*$ 可微：
$$\nabla f(\mathbf{x}^*) = \mathbf{0}$$
（"驻点"）

### 二阶充分条件 / Second-order sufficient

若 $\nabla f(\mathbf{x}^*) = \mathbf{0}$ **且** $\mathbf{H}(\mathbf{x}^*) \succ 0$（正定），则 $\mathbf{x}^*$ 是**严格局部极小**。
A stationary point with positive-definite Hessian is a strict local min.

### 凸函数的特例 / Convex case

$f$ 凸 ⇒ 一阶条件**也是充分的**：
$$\nabla f(\mathbf{x}^*) = \mathbf{0} \;\iff\; \mathbf{x}^* \text{ 是全局最优}$$


<a id="4"></a>
## 4. 拉格朗日 & KKT ⭐ / Lagrangian & KKT

### 4.1 等式约束 / Equality only

问题：$\min f(\mathbf{x})$ s.t. $h_j(\mathbf{x}) = 0$。

**拉格朗日函数 / Lagrangian**：
$$\mathcal{L}(\mathbf{x}, \boldsymbol{\nu}) = f(\mathbf{x}) + \sum_j \nu_j\,h_j(\mathbf{x})$$

最优条件 / Optimality:
$$\nabla_{\mathbf{x}} \mathcal{L} = \mathbf{0}, \qquad h_j(\mathbf{x}) = 0\; \forall j$$

### 4.2 完整约束 → KKT 条件

加上不等式约束 $g_i \le 0$：
$$\mathcal{L}(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\nu}) = f(\mathbf{x}) + \sum_i \lambda_i\,g_i(\mathbf{x}) + \sum_j \nu_j\,h_j(\mathbf{x})$$

$\boldsymbol{\lambda} \ge 0$ 是不等式的乘子。
$\boldsymbol{\nu}$ 是等式的乘子（无符号约束）。

**KKT 条件** / KKT conditions:
$$
\begin{aligned}
\text{(1) Stationarity:}\;\; & \nabla_{\mathbf{x}} \mathcal{L} = \mathbf{0} \\
\text{(2) Primal feasibility:}\;\; & g_i(\mathbf{x}) \le 0, \; h_j(\mathbf{x}) = 0 \\
\text{(3) Dual feasibility:}\;\; & \lambda_i \ge 0 \\
\text{(4) Complementary slackness:}\;\; & \lambda_i\, g_i(\mathbf{x}) = 0 \;\forall i
\end{aligned}
$$

> 💡 **互补松弛 / Complementary slackness** 一句话：每个不等式约束**要么严格成立** ($g_i < 0$) 让 $\lambda_i = 0$；**要么乘子非零** ($\lambda_i > 0$) 让 $g_i$ 等号成立。
> Each inequality is either inactive ($g_i < 0$, $\lambda_i = 0$) or active ($g_i = 0$, $\lambda_i > 0$).

### KKT 在 ML 里的两个出场 / KKT in ML

- **SVM 对偶**：KKT 推出"支持向量 = 让 $g_i = 0$ 的点"
- **Lasso**：subdifferential 形式的 KKT → coordinate descent 闭式更新
- SVM dual: the support vectors are exactly the points where the constraint is active.
- Lasso: subdifferential KKT gives closed-form coordinate updates.

### 例题 / Example

$\min x^2 + y^2$ s.t. $x + y = 1$。

$\mathcal{L} = x^2 + y^2 + \nu(x + y - 1)$。求偏导置 0：
$$2x + \nu = 0, \quad 2y + \nu = 0, \quad x + y = 1$$

$\Rightarrow x = y = 1/2$，$\nu = -1$。


In [ ]:
# 用 scipy 验证 / Verify with scipy
res = opt.minimize(
    lambda v: v[0]**2 + v[1]**2,
    x0=[0.0, 0.0],
    constraints={"type": "eq", "fun": lambda v: v[0] + v[1] - 1},
)
print(f"x* = {res.x.round(4)}    (expect [0.5, 0.5])")
print(f"f* = {res.fun:.4f}        (expect 0.5)")


<a id="5"></a>
## 5. Newton 法 / Newton's Method

### 想法 / Idea

二阶 Taylor 在当前点近似：
$$f(\mathbf{x} + \mathbf{d}) \approx f(\mathbf{x}) + \mathbf{g}^\top \mathbf{d} + \tfrac{1}{2}\mathbf{d}^\top \mathbf{H}\,\mathbf{d}$$

对 $\mathbf{d}$ 求最优（$\mathbf{H}$ 正定时）：
$$
\boxed{\;\mathbf{d}_{\text{Newton}} = -\mathbf{H}^{-1}\mathbf{g}\;}
$$

更新：$\mathbf{x}^{(t+1)} = \mathbf{x}^{(t)} - \mathbf{H}^{-1}\mathbf{g}$。

### 为什么"聪明"/ Why it's smart

Newton **预条件**了梯度：在曲率大的方向自动**缩短步长**，在曲率小的方向**放大步长**。GD 用统一 $\eta$，Newton 用按方向定制的 $\mathbf{H}^{-1}$。
Newton **preconditions** the gradient: short steps where curvature is high, long where it's low. GD uses one $\eta$; Newton uses a direction-aware $\mathbf{H}^{-1}$.

### 收敛速度 / Convergence rate

| 方法 / Method | 收敛 / Convergence |
|---|---|
| GD | **线性** / linear (loss ratio shrinks by constant factor each step) |
| Newton | **二次** / quadratic (number of correct digits roughly **doubles** per step) ⭐ |

但 Newton **每步代价**是 $O(n^3)$（求 Hessian + 解线性方程）。所以：

| 问题规模 / Problem size | 选谁 / Pick |
|---|---|
| $n < 10^3$ | **Newton** 直接秒杀 |
| $n \in [10^3, 10^6]$ | **L-BFGS** / 拟 Newton |
| $n > 10^6$（DL）| **一阶方法 (SGD/Adam)** —— Newton 内存爆炸 |

> 💡 **面试一句话答 / One-line interview**:
> "Newton would be ideal but it needs $O(n^2)$ memory and $O(n^3)$ per-step compute; modern DL has $n \sim 10^9$ parameters, so we fall back to first-order methods with curvature heuristics like Adam."


In [ ]:
# Newton vs GD 在 f(x,y) = x² + 25y² 上对比
# Newton vs GD on f(x,y) = x² + 25y² (ill-conditioned)
def f(p):     return p[0]**2 + 25*p[1]**2
def grad(p):  return np.array([2*p[0], 50*p[1]])
def hess(p):  return np.array([[2., 0.], [0., 50.]])

def gd(p0, eta, n_steps=40):
    ps = [p0.copy()]
    for _ in range(n_steps):
        ps.append(ps[-1] - eta * grad(ps[-1]))
    return np.array(ps)

def newton(p0, n_steps=10):
    ps = [p0.copy()]
    for _ in range(n_steps):
        ps.append(ps[-1] - np.linalg.solve(hess(ps[-1]), grad(ps[-1])))
    return np.array(ps)

start = np.array([5.0, 1.0])
gd_path = gd(start, eta=0.035)
newton_path = newton(start)

# 可视化 / Visualize
xs = np.linspace(-6, 6, 200); ys = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(xs, ys); Z = X**2 + 25*Y**2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, path, name in [(axes[0], gd_path, f"GD η=0.035, {len(gd_path)-1} steps"),
                        (axes[1], newton_path, f"Newton, {len(newton_path)-1} steps")]:
    ax.contour(X, Y, Z, levels=20, cmap="viridis", alpha=0.6)
    ax.plot(path[:, 0], path[:, 1], "o-", color="red", markersize=4)
    ax.scatter(0, 0, marker="*", color="green", s=200, zorder=5, label="optimum")
    ax.scatter(path[0, 0], path[0, 1], color="blue", s=80, zorder=5, label="start")
    ax.set_xlim(-6, 6); ax.set_ylim(-2, 2); ax.set_aspect("equal")
    ax.legend(); ax.set_title(name)
plt.tight_layout()
plt.show()

print(f"GD     final: {gd_path[-1]}    loss = {f(gd_path[-1]):.6f}")
print(f"Newton final: {newton_path[-1]}    loss = {f(newton_path[-1]):.6e}")


**看 Newton**：1 步就到原点！因为 $f$ 是**严格二次型**，Newton 用 $\mathbf{H}^{-1}$ 完美补偿了 $x$ 和 $y$ 方向曲率差异（1 vs 25）。
**Newton converges in 1 step** on a quadratic — $\mathbf{H}^{-1}$ exactly cancels the curvature mismatch.

**GD 反复震荡**：默认 $\eta=0.035$ 在 $y$ 方向（曲率 50）勉强稳定，但在 $x$ 方向（曲率 2）走得超慢。这就是**病态条件数问题** / ill-conditioning。
**GD ping-pongs** because the same $\eta$ has to work for two very different curvatures.


<a id="6"></a>
## 6. 拟 Newton：BFGS & L-BFGS / Quasi-Newton

**痛点**：Newton 要算并求逆 Hessian — DL 完全不可行。
**Pain**: Newton needs the Hessian inverse — infeasible for DL.

**解决**：用历史梯度信息**近似** $\mathbf{H}^{-1}$，永远不显式存它。这就是**拟 Newton**。
**Idea**: approximate $\mathbf{H}^{-1}$ from gradient history without ever materializing it.

### BFGS

存一个 $n \times n$ 的 $\mathbf{B}_t \approx \mathbf{H}^{-1}$；每步用最近的 $\mathbf{s} = \mathbf{x}^{(t+1)} - \mathbf{x}^{(t)}$ 和 $\mathbf{y} = \mathbf{g}^{(t+1)} - \mathbf{g}^{(t)}$ 做秩-2 更新。
**问题**：$\mathbf{B}_t$ 还是 $O(n^2)$ 内存。

### L-BFGS（"Limited memory" BFGS）

**不存** $\mathbf{B}_t$，只存最近 $m$ 步的 $(\mathbf{s}, \mathbf{y})$ 对（典型 $m = 10$）。每次用一个双循环算出 $\mathbf{B}_t \mathbf{g}$。
**Memory**: $O(mn)$，**linear**！

| 算法 / Algorithm | 内存 / Memory | 适用 / Use |
|---|---|---|
| Newton    | $O(n^2)$ | $n$ 小 |
| BFGS      | $O(n^2)$ | $n$ 中等 |
| L-BFGS    | $O(mn)$  | **大规模无约束** —— sklearn `solver='lbfgs'` 默认用 |
| L-BFGS-B  | 同上 + bound 约束 | scipy 默认 |

> 💡 **sklearn 的 `LogisticRegression(solver='lbfgs')`** 就是这个。在中小规模 + 凸损失下，L-BFGS 通常比 SGD/Adam 快得多。
> sklearn's default for logistic regression. Beats SGD/Adam on small-to-medium convex losses.


In [ ]:
# 用 scipy 的 L-BFGS 解 Rosenbrock 函数 / L-BFGS on Rosenbrock
def rosen(x):     return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2
def rosen_grad(x):
    return np.array([
        -2*(1 - x[0]) - 400*x[0]*(x[1] - x[0]**2),
        200*(x[1] - x[0]**2),
    ])

res = opt.minimize(rosen, x0=[-1.0, 1.5], jac=rosen_grad, method="L-BFGS-B")
print(f"converged in {res.nit} iterations")
print(f"x*    = {res.x.round(6)}    (expect [1, 1])")
print(f"f(x*) = {res.fun:.3e}")


<a id="7"></a>
## 7. 一阶方法谱系 / First-Order Optimizer Zoo

DL 的"主战场"。**所有现代 NN 优化器都是 GD 的某种变体**。
The DL workhorse. **All modern NN optimizers are variants of GD.**

下面每个的更新都是 $\mathbf{x}^{(t+1)} = \mathbf{x}^{(t)} - (\text{某种自适应步长}) \cdot (\text{某种梯度方向})$。
Each does $\mathbf{x}^{(t+1)} = \mathbf{x}^{(t)} - (\text{adaptive step}) \cdot (\text{some gradient direction})$.

### 7.1 SGD —— 随机梯度下降

不在整个数据集上求 $\nabla$，而在**小批量 mini-batch** 上估计：
$$\mathbf{x}^{(t+1)} = \mathbf{x}^{(t)} - \eta\,\hat{\mathbf{g}}^{(t)}$$
$\hat{\mathbf{g}}^{(t)}$ 是当前 mini-batch 的梯度。**噪声 = 助力**：帮助逃出鞍点。
The noise from mini-batches actually helps escape saddles.

### 7.2 Momentum / Polyak

把"上一次的速度"也加进来，平滑震荡：
$$\mathbf{v}^{(t+1)} = \beta \mathbf{v}^{(t)} + \mathbf{g}^{(t)}$$
$$\mathbf{x}^{(t+1)} = \mathbf{x}^{(t)} - \eta\,\mathbf{v}^{(t+1)}$$
$\beta \approx 0.9$ 是典型设置。
Like a ball rolling downhill with inertia — smooths out oscillations.

### 7.3 Nesterov Accelerated Gradient (NAG)

"先按动量走一步，再在那个点求梯度"——梯度评估"超前一步"：
$$\mathbf{v}^{(t+1)} = \beta \mathbf{v}^{(t)} + \nabla f(\mathbf{x}^{(t)} - \eta\beta \mathbf{v}^{(t)})$$
理论上对**凸**问题有更好的收敛率（$O(1/T^2)$ vs $O(1/T)$）。
Better theoretical rates on convex problems.

### 7.4 Adagrad

每个**参数**独立累计平方梯度做学习率自适应：
$$G^{(t)}_i = G^{(t-1)}_i + g^{(t)}_i{}^2, \quad x^{(t+1)}_i = x^{(t)}_i - \frac{\eta}{\sqrt{G^{(t)}_i} + \epsilon}\,g^{(t)}_i$$
**问题**：累积 → $G$ 永远递增 → 后期步长趋 0。
**Issue**: $G$ grows unboundedly → step size dies.

### 7.5 RMSProp

修正 Adagrad：用**指数滑动平均**而不是累计：
$$G^{(t)}_i = \rho G^{(t-1)}_i + (1-\rho) g^{(t)}_i{}^2$$
Hinton 在 Coursera 提出。$\rho \approx 0.99$。
Proposed by Hinton on Coursera. Fixes Adagrad's decay problem.

### 7.6 Adam ⭐

Momentum + RMSProp 缝合：
$$
\mathbf{m}^{(t)} = \beta_1 \mathbf{m}^{(t-1)} + (1-\beta_1) \mathbf{g}^{(t)} \quad\text{(动量)}
$$
$$
\mathbf{v}^{(t)} = \beta_2 \mathbf{v}^{(t-1)} + (1-\beta_2) \mathbf{g}^{(t)}{}^2 \quad\text{(平方平均)}
$$
**偏差校正** / Bias correction：
$$\hat{\mathbf{m}} = \mathbf{m}/(1-\beta_1^t), \quad \hat{\mathbf{v}} = \mathbf{v}/(1-\beta_2^t)$$
$$\mathbf{x}^{(t+1)} = \mathbf{x}^{(t)} - \eta\,\dfrac{\hat{\mathbf{m}}}{\sqrt{\hat{\mathbf{v}}} + \epsilon}$$

默认 $\beta_1 = 0.9, \beta_2 = 0.999, \eta = 10^{-3}$。**DL 默认 optimizer**。
Default in DL.

### 7.7 AdamW

把 weight decay 从梯度里**解耦**出来：
$$\mathbf{x}^{(t+1)} = (1 - \eta\lambda)\,\mathbf{x}^{(t)} - \eta\,\dfrac{\hat{\mathbf{m}}}{\sqrt{\hat{\mathbf{v}}} + \epsilon}$$
**训练 BERT / GPT 几乎全用它**。
Used in BERT / GPT training. Decouples weight decay from the gradient signal.

### 7.8 一句话决策 / One-line picking guide

| 场景 / Setting | 选谁 / Pick |
|---|---|
| 凸 + 中等规模 | **L-BFGS** |
| NN 训练通用 | **Adam** / AdamW |
| 视觉训练，调时间 | **SGD with momentum + cosine LR schedule** (state-of-art on ImageNet) |
| 稀疏数据 / NLP 旧风格 | Adagrad |


<a id="8"></a>
## 8. 学习率调度 / LR Schedulers

固定 $\eta$ 在实际训练中往往不够。常见调度：
A constant LR is rarely optimal. Common schedules:

| 调度 / Schedule | 公式 / Formula | 何时用 |
|---|---|---|
| **Step decay** | 每 $K$ 步乘 $\gamma$ | 简单视觉训练 |
| **Cosine** | $\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos(\pi t/T))$ | DL 主流，ResNet/Transformer 都用 |
| **Warmup + Cosine** | 前 $W$ 步线性 ramp 到 $\eta_{\max}$，然后 cosine 衰减 | **大模型训练标配** |
| **One-cycle** | 先涨后降 + 末尾大幅衰减 | Fast.ai 风格 |
| **ReduceLROnPlateau** | val loss 不下降就 $\times 0.1$ | 自适应 / 简单 |


In [ ]:
# 几种主要 schedule 可视化 / Visualize main schedules
steps = np.arange(0, 1000)
eta_max, eta_min, T = 1e-3, 1e-5, 1000

fig, ax = plt.subplots(figsize=(10, 4))

# step decay
step_lr = np.ones_like(steps, dtype=float) * eta_max
for k in range(1, 4):
    step_lr[steps >= k * 250] *= 0.5
ax.plot(steps, step_lr, label="step decay")

# cosine
cosine_lr = eta_min + 0.5*(eta_max-eta_min)*(1+np.cos(np.pi*steps/T))
ax.plot(steps, cosine_lr, label="cosine")

# warmup + cosine
warmup = 100
warmup_cosine = np.where(
    steps < warmup,
    eta_max * steps / warmup,
    eta_min + 0.5*(eta_max-eta_min)*(1+np.cos(np.pi*(steps-warmup)/(T-warmup))),
)
ax.plot(steps, warmup_cosine, label="warmup + cosine ⭐")

# exponential
expo_lr = eta_max * (0.001 / 1.0)**(steps/T)
ax.plot(steps, expo_lr, label="exponential")

ax.set_xlabel("training step")
ax.set_ylabel("learning rate")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Common learning-rate schedules")
plt.show()


> 💡 **大模型训练 (BERT / GPT) 默认 = warmup + cosine** —— 一开始用低 lr 让权重稳定，再用 cosine 平滑下降。
> Standard for BERT / GPT-style training.


<a id="9"></a>
## 9. 实战 1：Newton 法解 Logistic 回归

用 Newton 法手写 logistic regression，对比 GD。**这是面试白板题的"教科书"答案**。
Implement logistic regression with Newton's method and compare to GD. **The textbook whiteboard answer.**

### 数学 / Math

Logistic regression with $\sigma(z) = 1/(1+e^{-z})$，目标负对数似然：
$$J(\mathbf{w}) = -\frac{1}{n}\sum_i \bigl[y_i \log \sigma(\mathbf{x}_i^\top\mathbf{w}) + (1-y_i)\log(1 - \sigma(\mathbf{x}_i^\top\mathbf{w}))\bigr]$$

梯度（推导见 0.8 节小结）：
$$\nabla J(\mathbf{w}) = \frac{1}{n}\mathbf{X}^\top(\boldsymbol{\sigma} - \mathbf{y}), \quad \sigma_i = \sigma(\mathbf{x}_i^\top \mathbf{w})$$

Hessian（关键推导）：
$$\mathbf{H}(\mathbf{w}) = \frac{1}{n}\mathbf{X}^\top \mathbf{S}\, \mathbf{X}, \quad \mathbf{S} = \mathrm{diag}\bigl(\sigma_i(1 - \sigma_i)\bigr)$$

Newton update:
$$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \mathbf{H}^{-1}\nabla J$$


In [ ]:
# 数据 / Data
from sklearn.datasets import load_breast_cancer
ds = load_breast_cancer()
X = ds.data
y = ds.target               # 0 = malignant, 1 = benign

# 标准化 + 加 bias / Standardize + bias
X = (X - X.mean(0)) / X.std(0)
X = np.hstack([np.ones((X.shape[0], 1)), X])
n, d = X.shape
print(f"n={n}, d={d}")


In [ ]:
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def neg_log_lik(w, X, y):
    # Numerically stable BCE: log(1+exp(-|z|)) + max(0, -z) - z*y
    z = X @ w
    return float(np.mean(np.log1p(np.exp(-np.abs(z))) + np.maximum(0, -z) + z - z*y))

def grad(w, X, y):
    return X.T @ (sigmoid(X @ w) - y) / len(y)

def hess(w, X, y):
    s = sigmoid(X @ w)
    S = s * (1 - s)
    return (X.T * S) @ X / len(y)


In [ ]:
# Newton 法 / Newton's method
w_newton = np.zeros(d)
newton_losses = []
for t in range(20):
    newton_losses.append(neg_log_lik(w_newton, X, y))
    H = hess(w_newton, X, y) + 1e-6 * np.eye(d)   # tiny reg for numerical stability
    g = grad(w_newton, X, y)
    w_newton = w_newton - np.linalg.solve(H, g)

print(f"Newton final loss: {newton_losses[-1]:.6f} after {len(newton_losses)} iters")


In [ ]:
# Gradient descent / GD
w_gd = np.zeros(d)
gd_losses = []
eta = 0.5
for t in range(200):
    gd_losses.append(neg_log_lik(w_gd, X, y))
    w_gd = w_gd - eta * grad(w_gd, X, y)

print(f"GD     final loss: {gd_losses[-1]:.6f} after {len(gd_losses)} iters")


In [ ]:
# 收敛对比 / Convergence comparison
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(newton_losses, "ro-", label="Newton (20 iters)", markersize=5)
ax.plot(gd_losses, "b.-", label="GD (200 iters)", markersize=3, alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel("negative log-likelihood")
ax.set_title("Logistic regression: Newton converges quadratically; GD linearly")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# 训练集准确率 / Train accuracy
pred_newton = (sigmoid(X @ w_newton) > 0.5).astype(int)
print(f"\nNewton train accuracy: {(pred_newton == y).mean():.4f}")


**Newton 法 4-5 步就到了 GD 200 步都达不到的精度** —— "迭代次数少几个数量级" 是 Newton 的核心卖点。
**Newton in ~5 iterations reaches an accuracy GD can't hit in 200** — orders-of-magnitude fewer iterations is Newton's signature.

但每一步代价是 $O(d^3)$（解 $\mathbf{H}^{-1}\mathbf{g}$），这就是为什么 DL 几百万参数时必须放弃 Newton。
But each step is $O(d^3)$ — infeasible at DL scale.


<a id="10"></a>
## 10. 实战 2：六种优化器在 2D 函数上对比 / Optimizer Race

在病态二维问题 $f(x,y) = x^2 + 25y^2$ 上让 6 个优化器**赛跑**。
Race six optimizers on the ill-conditioned $f(x,y) = x^2 + 25y^2$.


In [ ]:
def f(p): return p[0]**2 + 25*p[1]**2
def g(p): return np.array([2*p[0], 50*p[1]])

def sgd(start, eta=0.015, T=80):
    p = start.copy(); path = [p.copy()]
    for _ in range(T):
        p -= eta * g(p); path.append(p.copy())
    return np.array(path)

def momentum_opt(start, eta=0.015, beta=0.9, T=80):
    p = start.copy(); v = np.zeros_like(p); path = [p.copy()]
    for _ in range(T):
        v = beta*v + g(p); p -= eta*v; path.append(p.copy())
    return np.array(path)

def nesterov(start, eta=0.015, beta=0.9, T=80):
    p = start.copy(); v = np.zeros_like(p); path = [p.copy()]
    for _ in range(T):
        v = beta*v + g(p - eta*beta*v); p -= eta*v; path.append(p.copy())
    return np.array(path)

def adagrad(start, eta=0.5, eps=1e-8, T=80):
    p = start.copy(); G = np.zeros_like(p); path = [p.copy()]
    for _ in range(T):
        gp = g(p); G += gp**2
        p -= eta * gp / (np.sqrt(G) + eps); path.append(p.copy())
    return np.array(path)

def rmsprop(start, eta=0.1, rho=0.9, eps=1e-8, T=80):
    p = start.copy(); G = np.zeros_like(p); path = [p.copy()]
    for _ in range(T):
        gp = g(p); G = rho*G + (1-rho)*gp**2
        p -= eta * gp / (np.sqrt(G) + eps); path.append(p.copy())
    return np.array(path)

def adam(start, eta=0.1, b1=0.9, b2=0.999, eps=1e-8, T=80):
    p = start.copy(); m = np.zeros_like(p); v = np.zeros_like(p); path = [p.copy()]
    for t in range(1, T+1):
        gp = g(p); m = b1*m + (1-b1)*gp; v = b2*v + (1-b2)*gp**2
        mh = m / (1 - b1**t); vh = v / (1 - b2**t)
        p -= eta * mh / (np.sqrt(vh) + eps); path.append(p.copy())
    return np.array(path)

start = np.array([5.0, 1.0])
T = 80
runs = {
    "SGD":      sgd(start.copy(),      T=T),
    "Momentum": momentum_opt(start.copy(), T=T),
    "Nesterov": nesterov(start.copy(),  T=T),
    "Adagrad":  adagrad(start.copy(),   T=T),
    "RMSProp":  rmsprop(start.copy(),   T=T),
    "Adam":     adam(start.copy(),      T=T),
}

# 等高线 + 路径 / Contour + paths
xs = np.linspace(-6, 6, 200); ys = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(xs, ys); Z = X**2 + 25*Y**2

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (name, path) in zip(axes.flat, runs.items()):
    ax.contour(X, Y, Z, levels=20, cmap="viridis", alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], "r.-", linewidth=1, markersize=3)
    ax.scatter(0, 0, marker="*", color="green", s=150, zorder=5)
    ax.scatter(path[0, 0], path[0, 1], color="blue", s=60, zorder=5)
    ax.set_xlim(-6, 6); ax.set_ylim(-2, 2); ax.set_aspect("equal")
    ax.set_title(f"{name}    final loss={f(path[-1]):.2e}")
plt.tight_layout()
plt.show()


In [ ]:
# 损失收敛曲线 / Loss curves
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, path in runs.items():
    losses = [f(p) for p in path]
    ax.plot(losses, label=name)
ax.set_yscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel("f(x, y)")
ax.legend()
ax.set_title("Optimizer race: loss vs iteration on ill-conditioned quadratic")
ax.grid(alpha=0.3)
plt.show()


**观察 / Observations**:
- **SGD** 在 $y$ 方向被 $\eta$ 卡死，慢
- **Momentum / Nesterov**：在主方向加速，但还是被病态条件数限制
- **Adagrad**：早期猛，但 $G$ 太大后期变慢
- **RMSProp / Adam**：**最像 Newton** —— 用平方梯度估计曲率，给每个维度独立缩放，**显著优于 SGD**

This is why **Adam dominates DL**: it approximates per-dimension curvature using cheap first-order info, getting Newton-like steps at SGD-like cost.


<a id="11"></a>
## 11. 小结 / Summary

### 概念地图 / Concept map

```
优化问题
  │
  ├── 凸性 ⭐
  │     ├── 凸集、凸函数定义
  │     ├── ML 损失谁凸谁不凸（LinReg/LogReg/SVM 凸；NN 不凸）
  │     └── 凸 ⇒ 一阶条件充分 ⇒ 局部最优 = 全局最优
  │
  ├── 无约束 → 一阶条件 ∇f=0, 二阶 H≻0
  │
  ├── 约束
  │     ├── 等式 → 拉格朗日 + ν
  │     └── 不等式 → KKT (stationarity + feasibility + dual + complementary slackness) ⭐
  │
  ├── 二阶方法
  │     ├── Newton —— x^{t+1} = x^t - H⁻¹ g  (二次收敛、O(n³))
  │     └── BFGS / L-BFGS —— 不直接存 H⁻¹
  │
  └── 一阶方法（DL 主战场）
        ├── SGD
        ├── + Momentum (β·v + g)
        ├── + Nesterov (lookahead momentum)
        ├── Adagrad → RMSProp → Adam → AdamW ⭐
        │
        └── LR Schedulers: cosine, warmup+cosine, ReduceOnPlateau
```

### 🧠 必背公式 / Must-know formulas

| 算法 | 更新 |
|---|---|
| GD | $\mathbf{x}^{t+1} = \mathbf{x}^t - \eta\mathbf{g}$ |
| Newton | $\mathbf{x}^{t+1} = \mathbf{x}^t - \mathbf{H}^{-1}\mathbf{g}$ |
| Momentum | $\mathbf{v}^{t+1} = \beta\mathbf{v}^t + \mathbf{g};\;\mathbf{x}^{t+1} = \mathbf{x}^t - \eta\mathbf{v}^{t+1}$ |
| Adam | $\mathbf{m}, \mathbf{v}$ 各自 EWMA; bias-correct; $\mathbf{x}^{t+1} -= \eta\hat{\mathbf{m}}/(\sqrt{\hat{\mathbf{v}}}+\epsilon)$ |
| AdamW | Adam + weight decay 解耦到 $\mathbf{x}$ |

KKT 四条件:
- Stationarity: $\nabla_{\mathbf{x}}\mathcal{L} = 0$
- Primal feasibility: $g_i \le 0,\, h_j = 0$
- Dual feasibility: $\lambda_i \ge 0$
- Complementary slackness: $\lambda_i g_i = 0$

### 💡 工业速查 / Industry cheat sheet

```python
# 凸 + 中小规模
scipy.optimize.minimize(f, x0, jac=grad, method="L-BFGS-B")
sklearn.linear_model.LogisticRegression(solver="lbfgs")   # 默认！

# DL 通用
torch.optim.Adam(params, lr=1e-3, betas=(0.9, 0.999))
torch.optim.AdamW(params, lr=1e-4, weight_decay=0.01)     # LLM 标配

# 视觉 SOTA
torch.optim.SGD(params, lr=0.1, momentum=0.9, nesterov=True)
torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=...)
```

### 💡 面试速查 / Interview must-knows

1. **凸函数定义 + 充分性结论**：弦在图像之上 ⇒ 局部 = 全局
2. **KKT 四条件**：完整背下来
3. **Newton 二次收敛 vs GD 线性**：精度每步翻倍
4. **DL 不用 Newton 因为** $O(n^2)$ 内存 + $O(n^3)$ 每步
5. **Adam = Momentum + RMSProp + 偏差校正**
6. **AdamW 修正了 Adam 里 L2 正则被自适应步长稀释**
7. **训练 LLM 标配**：AdamW + warmup + cosine

### 下一节预告 / Next up

**Part 0.11 · 信息论** —— 熵、KL 散度、互信息、交叉熵。**深度学习里所有"对齐 / 蒸馏 / 编码"** 都用它们。
**Part 0.11 · Information Theory** — entropy, KL, mutual info, cross-entropy. The currency of alignment / distillation / coding in DL.
